# All data together for QGIS — Coimbra district

**Purpose.** Put every dataset we have, already cleaned, in one
GeoPackage to look at them together in QGIS:
`data/interim/spatial/qgis_overview.gpkg`. No analysis here.

**Needs** the cleaned files from `02_citizen_science/` and
`03_expert_reference/` (run those first) and the ICNF burned areas
(downloaded by `02_citizen_science/acacia_observations.ipynb`).

**Layers** (EPSG:3763):
- `distrito_coimbra`: district outline (iNaturalist place 8422);
- `inat_acacia`, `inat_outras_plantas`: cleaned iNaturalist
  observations;
- `especialistas_poligonos`: cleaned expert polygons (colour by
  `use`);
- `areas_ardidas`: ICNF burned areas 1975–2025 that intersect the
  district (colour by `Ano`).

COS 2023 is not copied: open `data/interim/spatial/COS2023v1-S2.gpkg`
directly (it has the official style).

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests
from shapely.geometry import shape

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src.burned_areas import load_burned_areas

OUT = ROOT / "data" / "interim" / "spatial" / "qgis_overview.gpkg"
OUT.unlink(missing_ok=True)

place = requests.get("https://api.inaturalist.org/v1/places/8422", timeout=60).json()["results"][0]
district = gpd.GeoDataFrame({"nome": ["Distrito de Coimbra"]}, geometry=[shape(place["geometry_geojson"])], crs=4326).to_crs(3763)
district.to_file(OUT, layer="distrito_coimbra")

for name, layer in [("inat_acacia_clean.csv", "inat_acacia"), ("inat_other_plants_clean.csv", "inat_outras_plantas")]:
    df = pd.read_csv(ROOT / "data" / "interim" / "citizen_science" / name)
    gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs=4326).to_crs(3763).to_file(OUT, layer=layer)
    print(layer, len(df))

expert = gpd.read_file(ROOT / "data" / "interim" / "expert_reference" / "expert_polygons_clean.gpkg")
expert.to_file(OUT, layer="especialistas_poligonos")
print("especialistas_poligonos", len(expert))

fires = load_burned_areas(ROOT / "data" / "external" / "auxiliary" / "icnf_burned_areas")
fires = fires[fires.intersects(district.geometry.iloc[0])]
fires = fires[[c for c in ["Ano", "DH_Inicio", "AREA_HA", "area_ht"] if c in fires.columns] + ["geometry"]].copy()
fires["Ano"] = pd.to_numeric(fires["Ano"], errors="coerce")
fires["DH_Inicio"] = fires["DH_Inicio"].astype(str)
fires.to_file(OUT, layer="areas_ardidas")
print("areas_ardidas", len(fires))